# 02 - Variational AutoEncoder (VAE)

Trains a VAE on the **training split** to generate synthetic butterfly images.
Generated images are then added to the training set and the Baseline CNN is re-trained to measure the improvement.

## 1. Setup & Imports

In [1]:
import os, sys, json, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from PIL import Image

import torch
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as T
import torchvision.utils as vutils

from sklearn.metrics import accuracy_score, f1_score
import kagglehub

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

if torch.cuda.is_available():    device = torch.device("cuda")
elif torch.backends.mps.is_available(): device = torch.device("mps")
else: device = torch.device("cpu")
print("Device:", device)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
for c in [os.path.join(NOTEBOOK_DIR,"../src"), os.path.join(NOTEBOOK_DIR,"src"),
          "/Applications/Universidade/4ano_2semestre/ACA/projeto2/aml-butterfly-generative-augmentation/src"]:
    c = os.path.abspath(c)
    if os.path.isdir(c) and c not in sys.path:
        sys.path.append(c); break

from dataset import ButterflyDataset
from transforms import get_transforms
from models import BaselineCNN
from autoencoder import VAE, vae_loss
from utils import get_splits, get_class_mapping, GLOBAL_SEED
print("Modules imported!")

Device: mps
Modules imported!


/Users/matildecarvalho/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 2. Load Data (same canonical split)

In [2]:
path = kagglehub.competition_download('aca-tp-2')
train_dir = os.path.join(path, "train")
df = pd.read_csv(os.path.join(path, "train.csv"))[["filename", "label"]]

train_df, val_df, test_df = get_splits(df, seed=GLOBAL_SEED)
class_to_idx, idx_to_class, classes = get_class_mapping(df)
NUM_CLASSES = len(classes)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)} | Classes: {NUM_CLASSES}")

Train: 3327 | Val: 832 | Test: 1040 | Classes: 75


## 3. VAE DataLoader

For the VAE we use a **simple transform** (resize + normalise to [-1,1] to match the `tanh` output).

In [3]:
IMAGE_SIZE = 64
BATCH_SIZE = 64

# Normalise to [-1, 1] — required because the VAE decoder uses Tanh
vae_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(0.5),
    T.ToTensor(),
    T.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

vae_dataset = ButterflyDataset(df=train_df, img_dir=train_dir, transform=vae_transform)
vae_loader  = data.DataLoader(vae_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

print(f"VAE batches: {len(vae_loader)}")

VAE batches: 52


## 4. Initialise VAE

In [4]:
LATENT_DIM = 128
BETA = 1.0          # β-VAE weight on KL term (1 = standard VAE)

vae = VAE(latent_dim=LATENT_DIM).to(device)
vae_optimizer = optim.Adam(vae.parameters(), lr=1e-3)

print(f"VAE parameters: {sum(p.numel() for p in vae.parameters()):,}")

VAE parameters: 2,958,595


## 5. VAE Training Loop

In [ ]:
VAE_EPOCHS = 50
history_vae = {"total": [], "recon": [], "kl": []}

best_vae_loss = float("inf")
os.makedirs("../outputs/models", exist_ok=True)
vae_model_path = "../outputs/models/vae_best.pth"

for epoch in range(VAE_EPOCHS):
    vae.train()
    total_loss = recon_loss = kl_loss = 0.0

    for imgs, _ in tqdm(vae_loader, desc=f"VAE Epoch {epoch+1}/{VAE_EPOCHS}", leave=False):
        imgs = imgs.to(device)
        vae_optimizer.zero_grad()
        recon, mu, log_var = vae(imgs)
        loss, r, kl = vae_loss(recon, imgs, mu, log_var, beta=BETA)
        loss.backward()
        vae_optimizer.step()
        total_loss += loss.item()
        recon_loss += r.item()
        kl_loss    += kl.item()

    n = len(vae_loader)
    history_vae["total"].append(total_loss / n)
    history_vae["recon"].append(recon_loss / n)
    history_vae["kl"].append(kl_loss / n)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:03d}/{VAE_EPOCHS} | "
              f"Total: {history_vae['total'][-1]:.4f} | "
              f"Recon: {history_vae['recon'][-1]:.4f} | "
              f"KL: {history_vae['kl'][-1]:.4f}")

    if history_vae["total"][-1] < best_vae_loss:
        best_vae_loss = history_vae["total"][-1]
        torch.save(vae.state_dict(), vae_model_path)

print("\nVAE training complete!")

VAE Epoch 1/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 2/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 3/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 4/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 5/50:   0%|          | 0/52 [00:00<?, ?it/s]

...

Epoch 010/50 | Total: 1024.1606 | Recon: 872.0227 | KL: 152.1379


VAE Epoch 11/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 12/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 13/50:   0%|          | 0/52 [00:00<?, ?it/s]

VAE Epoch 14/50:   0%|          | 0/52 [00:00<?, ?it/s]

## 6. VAE Loss Curve

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history_vae["total"], label="Total")
plt.plot(history_vae["recon"], label="Reconstruction")
plt.plot(history_vae["kl"],   label="KL Divergence")
plt.title("VAE Training Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss")
plt.legend(); plt.grid(True); plt.tight_layout()
plt.savefig("../outputs/vae_loss.png", dpi=150)
plt.show()

## 7. Qualitative Evaluation

### 7a. Reconstructions

In [ ]:
vae.load_state_dict(torch.load(vae_model_path, map_location=device))
vae.eval()

# Take one batch from the VAE loader
sample_imgs, _ = next(iter(vae_loader))
sample_imgs = sample_imgs[:8].to(device)

with torch.no_grad():
    recons = vae.reconstruct(sample_imgs)

# Denormalise [-1,1] → [0,1]
def denorm(t):
    return (t * 0.5 + 0.5).clamp(0, 1)

comparison = torch.cat([denorm(sample_imgs.cpu()), denorm(recons.cpu())])
grid = vutils.make_grid(comparison, nrow=8, padding=2)

plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Top: Original | Bottom: Reconstructed")
plt.axis("off")
plt.savefig("../outputs/vae_reconstructions.png", dpi=150, bbox_inches="tight")
plt.show()

### 7b. Generated Samples (sampled from prior)

In [ ]:
with torch.no_grad():
    generated = vae.generate(n=16, device=device)

grid = vutils.make_grid(denorm(generated.cpu()), nrow=8, padding=2)
plt.figure(figsize=(14, 4))
plt.imshow(grid.permute(1, 2, 0))
plt.title("Generated samples (z ~ N(0,I))")
plt.axis("off")
plt.savefig("../outputs/vae_generated_samples.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Generate Augmented Dataset

Generate synthetic images **per class** and save them to disk so the augmented CNN can be trained.

In [ ]:
N_PER_CLASS = 20     # how many extra images to generate per class
GEN_DIR = "../outputs/generated_vae"
os.makedirs(GEN_DIR, exist_ok=True)

vae.eval()
generated_records = []   # will become a DataFrame → augmented train set

for cls in tqdm(classes, desc="Generating images"):
    cls_dir = os.path.join(GEN_DIR, cls)
    os.makedirs(cls_dir, exist_ok=True)

    # Get real images for this class to condition latent sampling
    cls_df = train_df[train_df["label"] == cls].sample(
        min(N_PER_CLASS, len(train_df[train_df["label"] == cls])),
        random_state=SEED
    )

    # Encode real images → get mu → decode from slightly perturbed z
    transform_plain = T.Compose([
        T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        T.ToTensor(),
        T.Normalize([0.5,0.5,0.5],[0.5,0.5,0.5]),
    ])

    imgs_list = []
    for fname in cls_df["filename"]:
        img = Image.open(os.path.join(train_dir, fname)).convert("RGB")
        imgs_list.append(transform_plain(img))

    imgs_tensor = torch.stack(imgs_list).to(device)

    with torch.no_grad():
        mu, log_var = vae.encoder(imgs_tensor)
        # Add small noise to mu for variety
        z = mu + 0.3 * torch.randn_like(mu)
        gen_imgs = vae.decoder(z)

    # Save generated images
    gen_imgs = denorm(gen_imgs.cpu())
    for i, img_tensor in enumerate(gen_imgs):
        fname = f"gen_{cls}_{i:04d}.png"
        save_path = os.path.join(cls_dir, fname)
        T.ToPILImage()(img_tensor).save(save_path)
        generated_records.append({"filename": os.path.join(cls, fname), "label": cls})

gen_df = pd.DataFrame(generated_records)
print(f"Generated {len(gen_df)} images across {len(classes)} classes.")

## 9. Re-train Baseline CNN with Augmented Data

In [ ]:
import torch.nn as nn
from torch.utils.data import ConcatDataset

train_transform, val_transform = get_transforms()

# Original training set
orig_dataset = ButterflyDataset(df=train_df, img_dir=train_dir, transform=train_transform)

# Generated images dataset (uses the same class→idx mapping)
gen_dataset  = ButterflyDataset(df=gen_df, img_dir=GEN_DIR, transform=train_transform)

# Combined
aug_dataset = ConcatDataset([orig_dataset, gen_dataset])

val_dataset  = ButterflyDataset(df=val_df,  img_dir=train_dir, transform=val_transform)
test_dataset = ButterflyDataset(df=test_df, img_dir=train_dir, transform=val_transform)

aug_loader  = data.DataLoader(aug_dataset,  batch_size=32, shuffle=True,  num_workers=0)
val_loader  = data.DataLoader(val_dataset,  batch_size=32, shuffle=False, num_workers=0)
test_loader = data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

print(f"Augmented train: {len(aug_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
EPOCHS = 20
model_aug = BaselineCNN(num_classes=NUM_CLASSES).to(device)
criterion  = nn.CrossEntropyLoss()
optimizer  = optim.Adam(model_aug.parameters(), lr=1e-3)

history_aug = {"train_loss":[], "val_loss":[], "train_acc":[], "val_acc":[], "val_f1":[]}
best_f1 = 0.0
aug_model_path = "../outputs/models/baseline_cnn_vae_aug_best.pth"

for epoch in range(EPOCHS):
    model_aug.train()
    trn_loss, preds_all, labels_all = 0.0, [], []
    for imgs, labels in tqdm(aug_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model_aug(imgs); loss = criterion(out, labels)
        loss.backward(); optimizer.step()
        trn_loss += loss.item() * imgs.size(0)
        preds_all.extend(out.argmax(1).cpu().numpy())
        labels_all.extend(labels.cpu().numpy())

    trn_loss /= len(aug_dataset)
    trn_acc   = accuracy_score(labels_all, preds_all)

    model_aug.eval()
    val_loss, preds_all, labels_all = 0.0, [], []
    with torch.no_grad():
        for imgs, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]", leave=False):
            imgs, labels = imgs.to(device), labels.to(device)
            out = model_aug(imgs)
            val_loss += criterion(out, labels).item() * imgs.size(0)
            preds_all.extend(out.argmax(1).cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

    val_loss /= len(val_dataset)
    val_acc   = accuracy_score(labels_all, preds_all)
    val_f1    = f1_score(labels_all, preds_all, average="weighted")

    history_aug["train_loss"].append(trn_loss)
    history_aug["train_acc"].append(trn_acc)
    history_aug["val_loss"].append(val_loss)
    history_aug["val_acc"].append(val_acc)
    history_aug["val_f1"].append(val_f1)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | Train Loss: {trn_loss:.4f}  Acc: {trn_acc:.4f} "
          f"|| Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model_aug.state_dict(), aug_model_path)
        print(f"   ⭐ Best model saved (Val F1: {best_f1:.4f})")

print("\nAugmented training complete!")

## 10. Final Test Evaluation & Comparison

In [ ]:
model_aug.load_state_dict(torch.load(aug_model_path, map_location=device))
model_aug.eval()

preds_all, labels_all = [], []
with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Test Evaluation"):
        preds_all.extend(model_aug(imgs.to(device)).argmax(1).cpu().numpy())
        labels_all.extend(labels.numpy())

aug_test_acc = accuracy_score(labels_all, preds_all)
aug_test_f1  = f1_score(labels_all, preds_all, average="weighted")

# Load baseline results for comparison
with open("../outputs/baseline_results.json") as f:
    baseline = json.load(f)

print("\n" + "="*60)
print("COMPARISON — Baseline vs VAE-Augmented CNN")
print("="*60)
print(f"{'Model':<35} {'Accuracy':>10} {'F1 (w.)':>10}")
print("-"*60)
print(f"{'Baseline CNN (original data)':<35} {baseline['test_accuracy']:>10.4f} {baseline['test_f1_weighted']:>10.4f}")
print(f"{'Baseline CNN + VAE augmentation':<35} {aug_test_acc:>10.4f} {aug_test_f1:>10.4f}")
print("="*60)

delta_acc = aug_test_acc - baseline["test_accuracy"]
delta_f1  = aug_test_f1  - baseline["test_f1_weighted"]
print(f"Δ Accuracy : {delta_acc:+.4f}")
print(f"Δ F1-Score : {delta_f1:+.4f}")

# Save results
with open("../outputs/vae_aug_results.json", "w") as f:
    json.dump({"model": "Baseline CNN + VAE augmentation",
               "test_accuracy": aug_test_acc,
               "test_f1_weighted": aug_test_f1}, f, indent=2)
print("\nResults saved → ../outputs/vae_aug_results.json")